#🧩 Step 1: Install Required Dependencies for the RAG Pipeline

In this step, we install and upgrade all the necessary Python libraries that power our Retrieval-Augmented Generation (RAG) workflow.
These packages handle everything from text extraction and chunking to embedding generation and vector-based retrieval.

In [1]:
# 🧩 Step 1: Install Required Dependencies for the RAG Pipeline

# This command ensures all required libraries are installed and updated.
# Each module plays a specific role in building and running the RAG workflow.
!pip -q install -U \
  langchain langchain-community==0.2.* langchain-openai \
  langchain-pinecone==0.1.* pinecone==4.* pypdf==4.* tiktoken langsmith[openai-agents]


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.4/214.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#🔐 Step 2: Configure Environment Variables and API Keys

In this step, we securely set up all the environment variables required to connect to LangSmith, OpenAI, and Pinecone.
These credentials are fetched directly from Google Colab’s userdata store to avoid exposing sensitive keys inside the notebook.

In [3]:
# 🔐 Step 2: Configure Environment Variables and API Keys

from google.colab import userdata
import os

# Enable tracing for LangSmith to monitor and debug LangChain runs
os.environ["LANGSMITH_TRACING"] = "true"

# Set LangSmith API endpoint and key for experiment tracking
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')

# Assign project name for grouping all traces under one dashboard
os.environ["LANGSMITH_PROJECT"] = "pr-abandoned-canvas-88"

# Retrieve OpenAI and Pinecone API keys securely from Colab’s secret storage
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')

# Define the embedding model and Pinecone configuration details
OPENAI_MODEL_EMB = "text-embedding-3-small"   # Embedding model (1536-dim output)
INDEX_NAME = "day3-rag-index"                 # Pinecone index name
PINECONE_CLOUD = "aws"                        # Cloud provider for Pinecone (AWS or GCP)
PINECONE_REGION = "us-east-1"
PINECONE_ENV = "us-east-1-aws"

             # Pinecone region (must match your account setup)


#🧠 Step 3: Import Core Libraries for RAG Pipeline

In this step, we import the essential libraries required to build the Retrieval-Augmented Generation (RAG) system.
These libraries handle file access, document loading, text splitting, embedding creation, and vector database management.

In [5]:
# 🧠 Step 3: Import Core Libraries for RAG Pipeline

# Standard utilities
from pathlib import Path            # For handling file and folder paths
import json                         # For reading and writing metadata
from typing import List             # For adding type hints (e.g., List[Document])

# LangChain integrations for OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # For embeddings and chat models

# Loaders to extract content from different document formats
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# Tool to split long documents into manageable chunks for embedding
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Core LangChain document representation (content + metadata)
from langchain.schema import Document

# Pinecone SDK and LangChain vector store wrapper
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore


#📂 Step 4: Load and Structure Core and Student Documents

In this step, we define helper functions to load, parse, and tag documents from both core reference materials and student folders.
Each file—whether it’s a PDF, text file, or JSON—is transformed into a LangChain Document object enriched with metadata like source, student, and category.
This metadata ensures that the retrieval stage can later filter and contextualize information accurately.

Logic Breakdown:

Core PDFs (projects.pdf, criteria.pdf, mentors.pdf) → Loaded once and tagged as category: core.

Student directories → Each contains a mix of:

report.pdf → Full project report.

summary.txt → Short overview (optional).

metadata.json → Structured student data (optional).

Each file type is tagged for traceability and organized in a single list of documents (raw_docs).

In [ ]:
# 📂 Step 4: Load and Structure Core and Student Documents

# Define the base data directory (ensure you’ve uploaded Day3_RAG to this path)
DATA = Path("/content/drive/MyDrive/APU TRAINING/TRAINING DATA")

# Define all core reference PDFs that apply to all students
CORE_PDFS = [
    DATA / "projects.pdf",
    DATA / "criteria.pdf",
    DATA / "mentors.pdf",
]

def load_core_pdfs() -> List[Document]:
    """
    Loads core reference PDFs such as 'projects', 'criteria', and 'mentors'.
    Each page is parsed as a LangChain Document and tagged with metadata.
    """
    docs = []
    for pdf in CORE_PDFS:
        if pdf.exists():
            # Load all pages in the PDF
            pages = PyPDFLoader(str(pdf)).load()
            # Add metadata identifying the file and category
            for p in pages:
                p.metadata.update({"source": pdf.name, "category": "core"})
            docs.extend(pages)
    return docs


def load_student_dirs() -> List[Document]:
    """
    Iterates through each student's folder and loads their files:
      - report.pdf → main project document
      - summary.txt → short text summary (optional)
      - metadata.json → structured student metadata (optional)
    All loaded files are tagged with the student’s name and file category.
    """
    docs = []
    students_dir = DATA / "students"
    if not students_dir.exists():
        return docs

    for student_dir in students_dir.iterdir():
        if not student_dir.is_dir():
            continue

        # --- Load student report (PDF) ---
        report_pdf = student_dir / "report.pdf"
        if report_pdf.exists():
            pages = PyPDFLoader(str(report_pdf)).load()
            for p in pages:
                p.metadata.update({
                    "source": f"{student_dir.name}/report.pdf",
                    "student": student_dir.name,
                    "category": "student_report"
                })
            docs.extend(pages)

        # --- Load student summary (TXT) ---
        summary_txt = student_dir / "summary.txt"
        if summary_txt.exists():
            tdocs = TextLoader(str(summary_txt), encoding="utf-8").load()
            for d in tdocs:
                d.metadata.update({
                    "source": f"{student_dir.name}/summary.txt",
                    "student": student_dir.name,
                    "category": "student_summary"
                })
            docs.extend(tdocs)

        # --- Load student metadata (JSON) ---
        meta_json = student_dir / "metadata.json"
        if meta_json.exists():
            try:
                meta = json.loads(meta_json.read_text(encoding="utf-8"))
                meta_doc = Document(
                    page_content=json.dumps(meta, ensure_ascii=False, indent=2),
                    metadata={
                        "source": f"{student_dir.name}/metadata.json",
                        "student": student_dir.name,
                        "category": "student_metadata"
                    }
                )
                docs.append(meta_doc)
            except Exception as e:
                print(f"Could not parse {meta_json}: {e}")

    return docs


# Combine all loaded core and student documents into a single collection
raw_docs = load_core_pdfs() + load_student_dirs()

# Display the total number of loaded pages + text documents
print(f"Loaded {len(raw_docs)} raw documents (pages + text).")


Loaded 131 raw documents (pages + text).


#✂️ Step 5: Split Documents into Manageable Chunks

In this step, we split all loaded documents into smaller, context-preserving chunks using LangChain’s RecursiveCharacterTextSplitter.
Chunking ensures that the text pieces are small enough for embeddings yet maintain meaningful context for accurate retrieval during RAG queries.

Key Parameters:

chunk_size=1000 → Maximum number of characters per chunk.

chunk_overlap=150 → Overlap between consecutive chunks to retain continuity.

separators → Defines the priority of how text is split — from larger structures (paragraphs) to smaller ones (characters).

After chunking, the code prints the total number of generated chunks and previews one example to verify that the process worked correctly.

In [ ]:
# ✂️ Step 5: Split Documents into Manageable Chunks

# Initialize the RecursiveCharacterTextSplitter with custom parameters
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,                # Each chunk will have up to 1000 characters
    chunk_overlap=150,              # Overlap between chunks to preserve sentence context
    separators=["\n\n", "\n", " ", ""],  # Logical order for splitting (paragraph > line > word > char)
)

# Apply the splitter to all raw documents
chunks = splitter.split_documents(raw_docs)

# Display total chunk count for confirmation
print(f"Created {len(chunks)} chunks.")

# Preview the content and metadata of the first chunk
print(chunks[0].page_content[:300], "...\n", chunks[0].metadata)


Created 282 chunks.
PROJECT
 
EVALUATION
 
CRITERIA
 
2024
 
 
GRADING
 
RUBRIC
 
 
TECHNICAL
 
EXCELLENCE
 
(40
 
points)
 
-
 
Code
 
Quality
 
and
 
Documentation:
 
10
 
pts
 
-
 
Algorithm
 
Efficiency:
 
10
 
pts
 
-
 
System
 
Architecture:
 
10
 
pts
 
-
 
Testing
 
and
 
Validation:
 
10
 
pts
 
 
INNOVATION
  ...
 {'source': 'criteria.pdf', 'page': 0, 'category': 'core'}


#🧮 Step 6: Generate Embeddings and Store Them in Pinecone

In this step, we transform our text chunks into numerical vector embeddings using OpenAI’s embedding model and store them in a Pinecone vector database.
This allows the RAG system to perform semantic search — retrieving the most contextually similar chunks for any query.

In [6]:
# 🧮 Step 6: Generate Embeddings and Store Them in Pinecone

# Initialize the OpenAI embedding model
# This model converts text chunks into 1536-dimensional numerical vectors
embeddings = OpenAIEmbeddings(model=OPENAI_MODEL_EMB)

# Connect to Pinecone using the stored API key
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Create a new Pinecone index if it doesn’t exist yet
# The dimension (1536) must match the output size of the embedding model
if INDEX_NAME not in [ix["name"] for ix in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,                 # Name of the index to create
        dimension=1536,                  # Embedding vector dimension
        metric="cosine",                 # Similarity metric for retrieval
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,        # Cloud provider (e.g., AWS or GCP)
            region=PINECONE_REGION       # Region where your Pinecone account is hosted
        ),
    )

# Build a LangChain vector store that wraps around the Pinecone index
# This uploads and organizes all document embeddings for retrieval
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,                    # The list of chunked documents
    embedding=embeddings,                # Embedding function
    index_name=INDEX_NAME,               # Pinecone index name
)


NameError: name 'chunks' is not defined

# If ingesion already done then do this instead

In [3]:
from google.colab import userdata
import os

# Standard utilities
from pathlib import Path            # For handling file and folder paths
import json                         # For reading and writing metadata
from typing import List             # For adding type hints (e.g., List[Document])

# LangChain integrations for OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # For embeddings and chat models

# Loaders to extract content from different document formats
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# Tool to split long documents into manageable chunks for embedding
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Core LangChain document representation (content + metadata)
from langchain.schema import Document

# Pinecone SDK and LangChain vector store wrapper
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# Enable LangSmith tracing for detailed observability of LangChain runs.
os.environ["LANGSMITH_TRACING"] = "true"

# Define the LangSmith API endpoint.
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

# Retrieve the LangSmith API key securely from Colab's secret store.
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')

# Assign a project name to organize runs within LangSmith.
os.environ["LANGSMITH_PROJECT"] = "RAG PROJECT"

# 2) Connect to existing index
INDEX_NAME = "day3-rag-index"   # <-- put your existing index name here
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))
index = pc.Index(INDEX_NAME)

# 3) Attach embeddings + vectorstore wrapper (NO re-indexing happens here)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = PineconeVectorStore(index=index, embedding=embeddings)


#💬 Step 7: Initialize the Chat Model and Create a Retriever

In this step, we set up the language model (LLM) that will generate answers and the retriever that will fetch relevant document chunks from Pinecone.
This forms the bridge between user questions and contextually relevant information stored in your RAG pipeline.

In [4]:
# 💬 Step 7: Initialize the Chat Model and Create a Retriever

# Initialize the LLM (Language Model)
# Using a compact and efficient OpenAI model for quick, low-cost reasoning
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # You can also use "gpt-4o-mini-2024-xx-xx"

# Create a retriever from the Pinecone vector store
# This retriever will fetch the top 'k' most relevant text chunks per user query
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


#🔄 Step 8: Implement Multi-Query Generation for Broader Retrieval Coverage

In this step, we add a multi-query generation mechanism that reformulates a single user question into several alternative versions.
This approach helps the retriever gather more comprehensive context from the vector database by accounting for variations in phrasing, synonyms, and perspective.

Why it matters:

Vector searches rely on similarity scores, so a single phrasing might miss semantically related documents.

By generating 5 alternative question variations, we increase recall and improve answer completeness.

In [5]:
# 🔄 Step 8: Implement Multi-Query Generation for Broader Retrieval Coverage

from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains import LLMChain
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema import StrOutputParser

# Define a prompt that instructs the LLM to create five alternative versions
# of the original user question — each phrased differently but semantically similar
template = """You are an AI language model assistant. Your task is to generate three
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines. Original question: {question}"""

# Convert raw text template into a ChatPromptTemplate
prompt_perspectives = ChatPromptTemplate.from_template(template)

# Create a multi-query generation chain
# The chain: (prompt → LLM → string output → list of questions)
generate_queries = (
    prompt_perspectives
    | ChatOpenAI(temperature=0)  # Deterministic model for consistent phrasing
    | StrOutputParser()          # Convert output to text
    | (lambda x: x.split("\n"))  # Split text by newlines into a list
)


#🔍 Step 9: Aggregate Unique Retrieved Documents from Multi-Query Search

In this step, we combine the multi-query generation with document retrieval to gather a comprehensive and unique set of relevant chunks from the vector database.
Each alternative version of the user’s question is used to perform a separate retrieval, and then all results are merged, deduplicated, and returned.

Why this matters:

Each phrasing of a query might capture slightly different context in the vector store.

Combining and deduplicating results ensures richer coverage without repetition.

In [6]:
# 🔍 Step 9: Aggregate Unique Retrieved Documents from Multi-Query Search

from langchain.load import dumps, loads

def get_unique_union(documents: list[list]):
    """Return a unique union of retrieved documents."""
    # Flatten nested lists and serialize each Document
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Remove duplicates by converting to a set
    unique_docs = list(set(flattened_docs))
    # Deserialize back into LangChain Document objects
    return [loads(doc) for doc in unique_docs]

# Example user query to demonstrate multi-query retrieval
question = (
    "Provide me names of 2 students which are best to work on a Robot that looks "
    "and air quality index and alert people if it's high. Provide reasons to select "
    "these two students too."
)

# Combine multi-query generation, retrieval, and deduplication
retrieval_chain = generate_queries | retriever.map() | get_unique_union

# Invoke the chain to retrieve unique relevant documents
docs = retrieval_chain.invoke({"question": question})

# Print how many unique documents were retrieved
len(docs)


/tmp/ipython-input-443419187.py:12: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


4

#🧩 Step 10: Build the Final RAG Chain for Context-Aware Answer Generation

In this step, we bring everything together to form the final RAG pipeline.
The system retrieves context using the multi-query retrieval chain built earlier and then passes that context to an OpenAI model to generate a coherent, fact-grounded answer.

In [7]:
# 🧩 Step 10: Build the Final RAG Chain for Context-Aware Answer Generation

from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

# Define a simple RAG-style prompt template
# The model is instructed to use only the retrieved context for answering
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

# Convert the text template into a ChatPromptTemplate object
prompt = ChatPromptTemplate.from_template(template)

# Initialize the LLM with deterministic output (temperature=0)
llm = ChatOpenAI(temperature=0)

# Combine the retriever, prompt, LLM, and output parser into one end-to-end chain
final_rag_chain = (
    {
        "context": retrieval_chain,          # Multi-query + deduplicated retrieval context
        "question": itemgetter("question")   # Pass original user question
    }
    | prompt                                 # Format the input as a structured prompt
    | llm                                    # Generate response using the LLM
    | StrOutputParser()                      # Extract and clean the string output
)

# Invoke the chain on the given question to generate the final grounded answer
final_rag_chain.invoke({"question": question})


'Based on the provided context, the two students best suited to work on a robot that monitors air quality and alerts people if it\'s high are David Miller and Dr. Emily Zhang.\n\n1. David Miller:\n- David\'s project title "Real-time Air Quality Monitoring Network with Predictive Analytics" demonstrates his expertise and interest in environmental monitoring and air quality.\n- His project involves IoT and predictive analytics, which are essential components for developing a robot that monitors air quality.\n- David\'s background in Environmental Science and experience in working with IoT technologies make him a suitable candidate for this project.\n\n2. Dr. Emily Zhang:\n- Dr. Emily Zhang\'s expertise in Robotics, Automation, and Computer Vision align well with the requirements of developing a robot for monitoring air quality.\n- Her experience in autonomous systems and computer vision can contribute to the design and functionality of the air quality monitoring robot.\n- Dr. Zhang\'s ba